# Artemus GPU Training — resumable foundation learning and instruction adaptation

This notebook runs one bounded, reproducible training cycle on a Kaggle GPU. The first cycle trains a foundation model from random initialization. Later cycles resume **only the foundation checkpoint**, preserve its tokenizer, and use a new deterministic streaming sample. Each cycle then creates a fresh instruction-tuned candidate from that cycle's foundation checkpoint.

For automated weekly runs, attach the private Kaggle dataset `artemus-training-state` as an input. It stores the foundation checkpoint and tokenizer between runs. The GitHub Actions workflow uploads the new checkpoint and provenance back into that dataset. Without that input dataset, the notebook deliberately starts a new experiment from random initialization.

This is a small research pilot, not frontier-scale training. FineWeb sources require their attribution, Common Crawl terms, and privacy caveats to be reviewed before model redistribution. Checkpoint promotion still requires separate held-out capability evaluation.

## 1. Setup — Kaggle GPU and persistent state

Run on a Kaggle notebook with a GPU and internet access. For recurring runs, attach the private `artemus-training-state` dataset. The notebook checks that its checkpoint and tokenizer agree before resuming. GPU availability is best-effort; a failed or interrupted run must not replace the last saved state.

In [ ]:
import torch
from pathlib import Path
import os, subprocess, json, shutil

if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU was allocated. Enable a GPU accelerator in Notebook settings, or use a CUDA VM.')

print('GPU:', torch.cuda.get_device_name(0))
print('CUDA:', torch.version.cuda)
print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
print('BF16 supported:', torch.cuda.is_bf16_supported())


## 2. Fetch the project and install the data tools

The notebook uses the public `main` branch as its code source. For a strict reproduction, copy the printed commit SHA into your experiment record and rerun with that exact repository commit.


In [ ]:
REPO_URL = 'https://github.com/FAHTHIZ/Artemus.git'
WORK_ROOT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/content')
WORKDIR = WORK_ROOT / 'Artemus-training'

# Never delete an existing workspace. Reuse only a clean clone of this repository.
if WORKDIR.exists():
    if not (WORKDIR / '.git').is_dir():
        raise RuntimeError(f'Workspace already exists and is not a Git clone: {WORKDIR}')
    origin = subprocess.check_output(
        ['git', 'remote', 'get-url', 'origin'], cwd=WORKDIR, text=True
    ).strip().removesuffix('.git').rstrip('/')
    expected_origin = REPO_URL.removesuffix('.git').rstrip('/')
    if origin.lower() != expected_origin.lower():
        raise RuntimeError(f'Existing workspace has a different Git origin: {origin}')
    dirty = subprocess.check_output(
        ['git', 'status', '--porcelain'], cwd=WORKDIR, text=True
    ).strip()
    if dirty:
        raise RuntimeError('Existing Artemus-training clone has local changes; preserve them and use a clean workspace.')
    subprocess.run(['git', 'pull', '--ff-only'], cwd=WORKDIR, check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(WORKDIR)], check=True)
os.chdir(WORKDIR)
REPO_COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Artemus commit:', REPO_COMMIT)
subprocess.run(['python', '-m', 'pip', 'install', '--upgrade', 'pip'], check=True)
subprocess.run(['python', '-m', 'pip', 'install', '-e', '.[dev,data]'], check=True)


## 3. Reproducible experiment settings

Keep the first run intentionally bounded. Increase document count and training steps only after the data pipeline, memory use, validation behavior, and checkpoint recovery are verified. `sample-10BT` names the upstream subset; this notebook does **not** download 10 billion tokens.


In [ ]:
DATASET_ID = 'HuggingFaceFW/fineweb-edu'
DATASET_CONFIG = 'sample-10BT'
DATASET_SPLIT = 'train'
DATASET_REVISION = '87f09149ef4734204d70ed1d046ddc9ca3f2b8f9'
DATASET_LICENSE = 'ODC-By-1.0'
LICENSE_REVIEW_NOTE = ('Research pilot only: upstream card declares ODC-By 1.0; '
    'Common Crawl terms and residual PII caveats require separate review before model redistribution.')

MAX_DOCUMENTS = 20_000
VALIDATION_PERCENT = 1.0
MIN_CHARS = 300
MAX_CHARS = 40_000
VOCAB_SIZE = 4096
BLOCK_SIZE = 512
TRAIN_STEPS = 2_000
BATCH_SIZE = 4
GRAD_ACCUM = 4
BASE_LR = 3e-4
PRECISION = 'bf16' if torch.cuda.is_bf16_supported() else 'fp16'

ARTIFACT_ROOT = WORK_ROOT / 'artifacts'
CHECKPOINT_ROOT = WORK_ROOT / 'checkpoints'
DATA_DIR = ARTIFACT_ROOT / 'fineweb_edu'
FINEWEB2_DIR = ARTIFACT_ROOT / 'fineweb2_bangla'
AYA_DIR = ARTIFACT_ROOT / 'aya_sft'
MIX_DIR = ARTIFACT_ROOT / 'foundation_mixture'
TOKENIZER_PREFIX = ARTIFACT_ROOT / 'artemus-foundation-tokenizer'
CHECKPOINT = CHECKPOINT_ROOT / 'artemus-foundation-fineweb-edu.pt'
DATA_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
FINEWEB2_MAX_DOCUMENTS = 5_000
MIX_TRAIN_DOCUMENTS = 10_000
MIX_VALIDATION_DOCUMENTS = 500
FINEWEB_ENGLISH_WEIGHT = 0.85
FINEWEB2_BANGLA_WEIGHT = 0.15
AYA_MAX_EXAMPLES = 10_000
AYA_SFT_STEPS = 1_000
# Persistent Kaggle state contains the last foundation checkpoint and its matching tokenizer.
# The first run sees only a bootstrap dataset and starts from random initialization.
STATE_DATASET_SLUG = 'artemus-training-state'
STATE_ROOT = Path('/kaggle/input') / STATE_DATASET_SLUG
STATE_CHECKPOINT = STATE_ROOT / 'artemus-foundation-current.pt'
STATE_TOKENIZER = STATE_ROOT / 'tokenizer.model'
STATE_MANIFEST = STATE_ROOT / 'state.json'
HAS_FOUNDATION_STATE = STATE_CHECKPOINT.is_file()
HAS_TOKENIZER_STATE = STATE_TOKENIZER.is_file()
if HAS_FOUNDATION_STATE != HAS_TOKENIZER_STATE:
    raise RuntimeError('Training state is incomplete: checkpoint and tokenizer must be present together.')
if HAS_FOUNDATION_STATE and not STATE_MANIFEST.is_file():
    raise RuntimeError('Training state has checkpoints but no state.json integrity manifest.')
if STATE_MANIFEST.is_file() and not HAS_FOUNDATION_STATE:
    raise RuntimeError('Training state has a manifest but is missing its foundation checkpoint.')

STATE_PAYLOAD = None
if HAS_FOUNDATION_STATE:
    import base64
    import hashlib

    STATE_RECORD = json.loads(STATE_MANIFEST.read_text(encoding='utf-8'))
    STATE_PAYLOAD = torch.load(STATE_CHECKPOINT, map_location='cpu', weights_only=True)
    if STATE_PAYLOAD.get('format_version') != 3:
        raise RuntimeError('Only provenance-enabled checkpoint format v3 can be resumed.')
    if STATE_PAYLOAD.get('response_only') is True:
        raise RuntimeError('Persistent state must contain a foundation checkpoint, not an instruction-SFT checkpoint.')
    STATE_METADATA = STATE_PAYLOAD.get('run_metadata') or {}
    STATE_LINEAGE = STATE_METADATA.get('lineage') or {}
    if STATE_LINEAGE.get('origin') != 'random_initialization':
        raise RuntimeError('Refusing to resume a checkpoint without verified random-initialization lineage.')

    checkpoint_sha = hashlib.sha256(STATE_CHECKPOINT.read_bytes()).hexdigest()
    if STATE_RECORD.get('foundation_checkpoint_sha256') != checkpoint_sha:
        raise RuntimeError('Foundation checkpoint SHA-256 does not match state.json; refusing corrupted state.')
    tokenizer_bytes = STATE_TOKENIZER.read_bytes()
    external_sha = hashlib.sha256(tokenizer_bytes).hexdigest()
    recorded_tokenizer_sha = STATE_RECORD.get('tokenizer_sha256')
    if recorded_tokenizer_sha is not None and recorded_tokenizer_sha != external_sha:
        raise RuntimeError('Tokenizer SHA-256 does not match state.json; refusing corrupted state.')

    tokenizer_b64 = STATE_PAYLOAD.get('tokenizer_model_b64')
    if not isinstance(tokenizer_b64, str):
        raise RuntimeError('Foundation checkpoint does not embed its SentencePiece tokenizer.')
    embedded_tokenizer = base64.b64decode(tokenizer_b64, validate=True)
    embedded_sha = hashlib.sha256(embedded_tokenizer).hexdigest()
    if embedded_sha != external_sha:
        raise RuntimeError('Persistent tokenizer does not match the tokenizer embedded in the foundation checkpoint.')

    CYCLE_INDEX = len(STATE_METADATA.get('runs') or [])
    if CYCLE_INDEX < 1:
        raise RuntimeError('Foundation checkpoint is missing its training-run lineage.')
    recorded_cycle = int(STATE_RECORD.get('training_cycle', -1))
    if recorded_cycle != CYCLE_INDEX:
        raise RuntimeError(
            f'Foundation checkpoint cycle ({CYCLE_INDEX}) does not match state.json ({recorded_cycle}).'
        )
else:
    CYCLE_INDEX = 0

FINEWEB_SHUFFLE_SEED = 1337 + CYCLE_INDEX * 104729
FINEWEB2_SHUFFLE_SEED = 2337 + CYCLE_INDEX * 104729
AYA_SHUFFLE_SEED = 3337 + CYCLE_INDEX * 104729
MIXTURE_SEED = 4337 + CYCLE_INDEX * 104729
TRAIN_SEED = 1337 + CYCLE_INDEX

print('Precision:', PRECISION)
print('Training cycle:', CYCLE_INDEX + 1)
print('Resume foundation checkpoint:', HAS_FOUNDATION_STATE)
print('Sampling seeds:', {
    'fineweb_edu': FINEWEB_SHUFFLE_SEED,
    'fineweb2_bengali': FINEWEB2_SHUFFLE_SEED,
    'aya': AYA_SHUFFLE_SEED,
    'mixture': MIXTURE_SEED,
})

TASK_SFT_STEPS = 750

### Data-use gate

The notebook records the source licence and review note in its manifest. The note is an audit entry, not legal advice or a substitute for reviewing the dataset card and upstream Common Crawl terms. If you have not accepted that research-use basis, stop before the download cell and choose another reviewed source.


In [ ]:
cmd = [
    'artemus-prepare-hf', '--dataset', DATASET_ID,
    '--config', DATASET_CONFIG, '--split', DATASET_SPLIT,
    '--revision', DATASET_REVISION, '--license', DATASET_LICENSE,
    '--license-review-note', LICENSE_REVIEW_NOTE,
    '--output-dir', str(DATA_DIR),
    '--max-documents', str(MAX_DOCUMENTS),
    '--validation-percent', str(VALIDATION_PERCENT),
    '--min-chars', str(MIN_CHARS), '--max-chars', str(MAX_CHARS),
    '--shuffle-seed', str(FINEWEB_SHUFFLE_SEED),
]
subprocess.run(cmd, check=True)
manifest = json.loads((DATA_DIR / 'manifest.json').read_text())
print(json.dumps(manifest['counts'], indent=2))
print('Accepted docs:', manifest['accepted_documents'])
print('Sampling:', manifest['sampling'])

## 4. Add a bounded Bangla stream and build a mixed foundation corpus\n\nThe Bengali FineWeb2 subset is streamed from its filtered `train` split only. The notebook resolves the current dataset commit once, writes that exact SHA into the source manifest, rejects obvious contact/secret patterns plus conservative phone-like sequences, and never downloads the full Bengali subset. The first model and tokenizer are trained on a deterministic 85/15 document mixture of FineWeb-Edu English and FineWeb2 Bangla. After tokenizer training, the notebook measures actual per-source token shares; they may differ from document weights because Bangla and English tokenize differently.\n\nThe dataset is ODC-By 1.0 and inherits Common Crawl terms. Its upstream maintainers warn that residual PII may remain and phone numbers are not comprehensively removed; our extra pattern filter is not a complete privacy audit. Review sampled documents and the resulting manifest before using or distributing any checkpoint.\n

In [ ]:
from huggingface_hub import HfApi

# Resolve the live dataset revision once; the actual training download is pinned to this SHA.
FINEWEB2_DATASET = 'HuggingFaceFW/fineweb-2'
fineweb2_revision = HfApi().dataset_info(FINEWEB2_DATASET).sha
if not fineweb2_revision or len(fineweb2_revision) < 7:
    raise RuntimeError('Could not resolve an immutable FineWeb2 dataset revision.')
FINEWEB2_LICENSE_NOTE = (
    'Pilot only: FineWeb2 card declares ODC-By 1.0; Common Crawl terms also apply. '
    'Upstream warns residual PII may remain and phone numbers are not comprehensively filtered. '
    'Artemus runs additional regex filtering, but this is not a complete privacy audit.'
)
fineweb2_cmd = [
    'artemus-prepare-hf', '--dataset', FINEWEB2_DATASET,
    '--config', 'ben_Beng', '--split', 'train', '--revision', fineweb2_revision,
    '--license', 'ODC-By-1.0', '--license-review-note', FINEWEB2_LICENSE_NOTE,
    '--output-dir', str(FINEWEB2_DIR), '--max-documents', str(FINEWEB2_MAX_DOCUMENTS),
    '--validation-percent', '1.0', '--min-chars', '300', '--max-chars', '40000',
    '--shuffle-seed', str(FINEWEB2_SHUFFLE_SEED),
]
subprocess.run(fineweb2_cmd, check=True)
fineweb2_manifest = json.loads((FINEWEB2_DIR / 'manifest.json').read_text(encoding='utf-8'))
fineweb2_manifest['resolved_revision'] = fineweb2_revision
(FINEWEB2_DIR / 'manifest.json').write_text(
    json.dumps(fineweb2_manifest, indent=2, ensure_ascii=False) + '\n', encoding='utf-8'
)
print('FineWeb2 revision:', fineweb2_revision)
print('Accepted Bangla documents:', fineweb2_manifest['accepted_documents'])
print('FineWeb2 sampling:', fineweb2_manifest['sampling'])


### Build the foundation mixture\n\nWe keep the split boundaries created by each source, choose deterministic document quotas, deduplicate across sources, and retain each record's original provenance. The 85/15 settings are a starting hypothesis—not a sacred recipe. Inspect the post-tokenizer shares before training and tune them from measured validation results.\n

In [ ]:
mixture_cmd = [
    'artemus-prepare-mixture', '--output-dir', str(MIX_DIR),
    '--train-documents', str(MIX_TRAIN_DOCUMENTS),
    '--validation-documents', str(MIX_VALIDATION_DOCUMENTS),
    '--seed', str(MIXTURE_SEED),
    '--source', 'fineweb-edu', str(DATA_DIR / 'train.jsonl'), str(DATA_DIR / 'validation.jsonl'), str(FINEWEB_ENGLISH_WEIGHT),
    '--source', 'fineweb2-ben_Beng', str(FINEWEB2_DIR / 'train.jsonl'), str(FINEWEB2_DIR / 'validation.jsonl'), str(FINEWEB2_BANGLA_WEIGHT),
]
subprocess.run(mixture_cmd, check=True)
mixture_manifest = json.loads((MIX_DIR / 'manifest.json').read_text(encoding='utf-8'))
print('Train mixture:', json.dumps(mixture_manifest['train'], indent=2, ensure_ascii=False))
print('Validation mixture:', json.dumps(mixture_manifest['validation'], indent=2, ensure_ascii=False))


## 4. Train a tokenizer on training text only

SentencePiece sees only the curated training documents, not validation. The model uses byte fallback and retains the dedicated chat control tokens for later supervised fine-tuning.


In [ ]:
from artemis.tokenizer import SentencePieceTokenizer
TOKENIZER_MODEL = TOKENIZER_PREFIX.with_suffix('.model')
if HAS_FOUNDATION_STATE:
    shutil.copy2(STATE_TOKENIZER, TOKENIZER_MODEL)
    tokenizer = SentencePieceTokenizer(model_file=TOKENIZER_MODEL)
else:
    tokenizer = SentencePieceTokenizer.train(
        MIX_DIR / 'train.txt', TOKENIZER_PREFIX, vocab_size=VOCAB_SIZE
    )
print('Tokenizer vocab:', tokenizer.vocab_size)
print('Tokenizer model:', TOKENIZER_PREFIX.with_suffix('.model'))


# Measure the real token mixture with the final tokenizer; document ratios are only a proxy.
from collections import Counter
mixture_token_counts = Counter()
for line in (MIX_DIR / 'train.jsonl').open('r', encoding='utf-8'):
    if line.strip():
        row = json.loads(line)
        mixture_token_counts[row['mixture']['source_id']] += len(tokenizer.encode(row['text']))
mixture_token_total = sum(mixture_token_counts.values())
mixture_token_shares = {
    key: value / mixture_token_total
    for key, value in mixture_token_counts.items()
}
mixture_manifest_path = MIX_DIR / 'manifest.json'
mixture_manifest = json.loads(mixture_manifest_path.read_text(encoding='utf-8'))
mixture_manifest['tokenizer_measurement'] = {
    'tokenizer': tokenizer.name,
    'vocab_size': tokenizer.vocab_size,
    'train_tokens_by_source': dict(mixture_token_counts),
    'train_token_share_by_source': mixture_token_shares,
}
mixture_manifest_path.write_text(json.dumps(mixture_manifest, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')
print('Actual token shares:', mixture_token_shares)


## 5. Continue foundation pretraining without resetting model weights

The first cycle starts from random initialization. Later cycles resume the previous **foundation** checkpoint, keep the exact same tokenizer and architecture, and train for another bounded segment on a newly shuffled sample. The separate instruction-tuned candidate is never fed back into foundation pretraining.

In [ ]:
# The state dataset contains only the foundation checkpoint, never the prior instruction-SFT candidate.
if HAS_FOUNDATION_STATE:
    shutil.copy2(STATE_CHECKPOINT, CHECKPOINT)
    previous_step = int(STATE_PAYLOAD['step'])
    target_step = previous_step + TRAIN_STEPS
    phase = 'continued_pretraining'
    resume_args = [
        '--resume', str(CHECKPOINT),
        '--allow-data-change',
        '--override-resume-learning-rate',
    ]
else:
    previous_step = 0
    target_step = TRAIN_STEPS
    phase = 'foundation_pretraining'
    resume_args = [
        '--tokenizer-model', str(TOKENIZER_PREFIX.with_suffix('.model')),
    ]

train_cmd = [
    'artemus-train', '--data', str(MIX_DIR / 'train.jsonl'),
    '--validation-data', str(MIX_DIR / 'validation.jsonl'),
    '--data-manifest', str(MIX_DIR / 'manifest.json'),
    '--document-format', 'jsonl', '--text-field', 'text',
    '--token-cache-dir', str(ARTIFACT_ROOT / 'token-cache'),
    '--tokenizer', 'sentencepiece',
    *resume_args,
    '--phase', phase, '--steps', str(target_step),
    '--batch-size', str(BATCH_SIZE), '--gradient-accumulation-steps', str(GRAD_ACCUM),
    '--precision', PRECISION, '--block-size', str(BLOCK_SIZE),
    '--layers', '4', '--heads', '4', '--embedding-size', '128', '--embedding-dim', '64',
    '--kv-heads', '2', '--use-rope', '--use-qk-norm', '--use-swiglu',
    '--learning-rate', str(BASE_LR), '--lr-schedule', 'cosine',
    '--min-learning-rate', '0.00003', '--warmup-steps', '100',
    '--eval-interval', '250', '--eval-iters', '8', '--seed', str(TRAIN_SEED),
    '--device', 'auto', '--output', str(CHECKPOINT),
]
print(f'Foundation cycle {CYCLE_INDEX + 1}: {phase}, steps {previous_step} -> {target_step}')
subprocess.run(train_cmd, check=True)
assert CHECKPOINT.is_file() and CHECKPOINT.stat().st_size > 0
foundation_payload = torch.load(CHECKPOINT, map_location='cpu', weights_only=True)
assert int(foundation_payload['step']) == target_step
assert foundation_payload.get('response_only') is False
print('Foundation checkpoint bytes:', CHECKPOINT.stat().st_size)

## 7. Optional code-domain continuation with OSCR

OSCR scripts are **not** under one universal licence. Each repository manifest preserves the source licence and the evidence used to confirm it. Only select repository manifest paths whose exact SPDX licence appears in the tool's allowlist (MIT, Apache-2.0, BSD, ISC, CC0, etc.). Exclude NC/SA/unknown/compound licences unless a separate policy has approved them. OSCR recommends downloading only selected manifests and blocks, not crawling its site.

Add exact paths from the OSCR dataset's `manifests/.../*.json` tree below. Leave the list empty to skip this stage. The dataset revision is pinned in the script.


In [ ]:
OSCR_MANIFEST_PATHS = []  # e.g. ['manifests/ab/<repository-manifest>.json']
OSCR_DIR = ARTIFACT_ROOT / 'oscr'
OSCR_MIX_DIR = ARTIFACT_ROOT / 'code_replay_mixture'
if OSCR_MANIFEST_PATHS:
    oscr_cmd = ['artemus-prepare-oscr', '--output-dir', str(OSCR_DIR),
                '--dataset-revision', 'b0e3fadb276678880f7f3d764308fbebd3bcd044',
                '--max-files', '10000']
    for manifest_path in OSCR_MANIFEST_PATHS:
        oscr_cmd.extend(['--hub-manifest-path', manifest_path])
    subprocess.run(oscr_cmd, check=True)

    # Retain the general multilingual foundation mixture during code adaptation.
    code_mix_cmd = [
        'artemus-prepare-mixture', '--output-dir', str(OSCR_MIX_DIR),
        '--train-documents', '10000', '--validation-documents', '500',
        '--seed', '1338',
        '--source', 'general-replay', str(MIX_DIR / 'train.jsonl'), str(MIX_DIR / 'validation.jsonl'), '0.85',
        '--source', 'python-code', str(OSCR_DIR / 'train.jsonl'), str(OSCR_DIR / 'validation.jsonl'), '0.15',
    ]
    subprocess.run(code_mix_cmd, check=True)
    parent_state = torch.load(CHECKPOINT, map_location='cpu', weights_only=True)
    code_target_step = int(parent_state['step']) + 3000
    CODE_CHECKPOINT = CHECKPOINT_ROOT / 'artemus-code-adapted.pt'
    subprocess.run([
        'artemus-train', '--data', str(OSCR_MIX_DIR / 'train.jsonl'),
        '--validation-data', str(OSCR_MIX_DIR / 'validation.jsonl'),
        '--data-manifest', str(OSCR_MIX_DIR / 'manifest.json'),
        '--document-format', 'jsonl', '--text-field', 'text',
        '--token-cache-dir', str(ARTIFACT_ROOT / 'oscr-token-cache'),
        '--tokenizer', 'sentencepiece',
        '--resume', str(CHECKPOINT), '--allow-data-change',
        '--phase', 'continued_pretraining', '--steps', str(code_target_step),
        '--batch-size', str(BATCH_SIZE), '--gradient-accumulation-steps', str(GRAD_ACCUM),
        '--precision', PRECISION, '--override-resume-learning-rate',
        '--learning-rate', '0.0001', '--lr-schedule', 'cosine', '--min-learning-rate', '0.00001',
        '--warmup-steps', '100', '--eval-interval', '250', '--eval-iters', '8',
        '--device', 'auto', '--output', str(CODE_CHECKPOINT),
    ], check=True)
    CHECKPOINT = CODE_CHECKPOINT
else:
    print('OSCR stage skipped: no repository manifests selected.')


## 8. Multilingual instruction tuning with human-annotated Aya data\n\nAfter foundation pretraining (and optional OSCR code adaptation), this stage performs response-only supervised fine-tuning on a bounded, pinned sample of Cohere's Aya Dataset. It contains human-annotated prompts and completions across 65 languages, including Bengali. The adapter discards annotator IDs and demographic fields; only the prompt, response, language label, annotation type, and source hashes are retained. Human annotations still need factual regression tests.\n

In [ ]:
from huggingface_hub import HfApi

AYA_DATASET = 'CohereLabs/aya_dataset'
aya_revision = HfApi().dataset_info(AYA_DATASET).sha
if not aya_revision or len(aya_revision) < 7:
    raise RuntimeError('Could not resolve an immutable Aya dataset revision.')
subprocess.run([
    'artemus-prepare-aya', '--revision', aya_revision,
    '--split', 'train', '--output-dir', str(AYA_DIR),
    '--max-examples', str(AYA_MAX_EXAMPLES), '--validation-percent', '2.0',
    '--shuffle-seed', str(AYA_SHUFFLE_SEED),
], check=True)
aya_manifest = json.loads((AYA_DIR / 'manifest.json').read_text(encoding='utf-8'))
aya_manifest['resolved_revision'] = aya_revision
(AYA_DIR / 'manifest.json').write_text(
    json.dumps(aya_manifest, indent=2, ensure_ascii=False) + '\n', encoding='utf-8'
)
print('Aya revision:', aya_revision)
print('Accepted examples:', aya_manifest['accepted_examples'])
print('Languages:', aya_manifest['examples_by_language_code'])
print('Aya sampling:', aya_manifest['sampling'])

parent_state = torch.load(CHECKPOINT, map_location='cpu', weights_only=True)
aya_target_step = int(parent_state['step']) + AYA_SFT_STEPS
AYA_CHECKPOINT = CHECKPOINT_ROOT / 'artemus-aya-instruction-sft.pt'
subprocess.run([
    'artemus-train', '--data', str(AYA_DIR / 'train.jsonl'),
    '--validation-data', str(AYA_DIR / 'validation.jsonl'),
    '--data-manifest', str(AYA_DIR / 'manifest.json'),
    '--document-format', 'jsonl', '--text-field', 'text',
    '--token-cache-dir', str(ARTIFACT_ROOT / 'aya-token-cache'),
    '--tokenizer', 'sentencepiece', '--resume', str(CHECKPOINT),
    '--allow-data-change', '--allow-objective-change',
    '--phase', 'instruction_sft', '--response-only', '--steps', str(aya_target_step),
    '--batch-size', str(BATCH_SIZE), '--gradient-accumulation-steps', str(GRAD_ACCUM),
    '--precision', PRECISION, '--override-resume-learning-rate',
    '--learning-rate', '0.00005', '--lr-schedule', 'cosine', '--min-learning-rate', '0.000005',
    '--warmup-steps', '50', '--eval-interval', '200', '--eval-iters', '8',
    '--device', 'auto', '--output', str(AYA_CHECKPOINT),
], check=True)
CHECKPOINT = AYA_CHECKPOINT
print('Current candidate checkpoint:', CHECKPOINT)


## 10. Task-specific supervised tuning: conversations, maths and Python

This stage reuses the repository's pinned OASST1, GSM8K and MBPP preparation code. OASST1 conversation trees remain grouped; GSM8K uses only its official training file; MBPP includes only approved training and validation IDs, excluding its prompt and protected test IDs.

The task candidate starts from this cycle's Aya-tuned checkpoint. It remains separate from the foundation checkpoint used by the next cycle. Passing the preparation pipeline or a dialogue health check does not prove mathematical or coding correctness; independent exact-answer and isolated code-test evaluations remain release requirements.

In [ ]:
from urllib.request import urlretrieve
from artemis.prepare_oasst1 import SOURCE_URL as OASST1_URL, EXPECTED_SHA256 as OASST1_SHA256
from artemis.prepare_capability_data import (
    GSM8K_URL, GSM8K_SHA256, MBPP_URL, MBPP_GIT_BLOB_SHA,
)

RAW_DIR = ARTIFACT_ROOT / 'raw-capability-data'
OASST_DIR = ARTIFACT_ROOT / 'oasst1'
CAPABILITY_DIR = ARTIFACT_ROOT / 'capability-sft'
RAW_DIR.mkdir(parents=True, exist_ok=True)

OASST_RAW = RAW_DIR / 'oasst1-ready.messages.jsonl.gz'
GSM8K_RAW = RAW_DIR / 'gsm8k-train.jsonl'
MBPP_RAW = RAW_DIR / 'mbpp-full.jsonl'

for url, path in ((OASST1_URL, OASST_RAW), (GSM8K_URL, GSM8K_RAW), (MBPP_URL, MBPP_RAW)):
    if not path.is_file():
        urlretrieve(url, path)
    if path.stat().st_size == 0:
        raise RuntimeError(f'Training source is empty: {path.name}')

import hashlib
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

if sha256_file(OASST_RAW) != OASST1_SHA256:
    raise RuntimeError('OASST1 archive SHA-256 mismatch.')
if sha256_file(GSM8K_RAW) != GSM8K_SHA256:
    raise RuntimeError('GSM8K official training file SHA-256 mismatch.')

mbpp_bytes = MBPP_RAW.read_bytes()
mbpp_header = b'blob ' + str(len(mbpp_bytes)).encode('ascii') + bytes([0])
if hashlib.sha1(mbpp_header + mbpp_bytes).hexdigest() != MBPP_GIT_BLOB_SHA:
    raise RuntimeError('MBPP Git blob SHA mismatch.')

# Expand the conversation sample gradually, with a bounded per-cycle limit.
OASST_MAX_EXAMPLES = min(5_000 + CYCLE_INDEX * 2_000, 30_000)
subprocess.run([
    'python', '-m', 'artemis.prepare_oasst1',
    '--source', str(OASST_RAW),
    '--output-dir', str(OASST_DIR),
    '--max-examples', str(OASST_MAX_EXAMPLES),
], check=True)

subprocess.run([
    'python', '-m', 'artemis.prepare_capability_data',
    '--oasst-train', str(OASST_DIR / 'train.txt'),
    '--oasst-validation', str(OASST_DIR / 'validation.txt'),
    '--gsm8k-train', str(GSM8K_RAW),
    '--mbpp-full', str(MBPP_RAW),
    '--output-dir', str(CAPABILITY_DIR),
], check=True)

capability_manifest = json.loads((CAPABILITY_DIR / 'manifest.json').read_text(encoding='utf-8'))
print(json.dumps(capability_manifest, indent=2, ensure_ascii=False))

# Start from the current Aya-tuned instruction checkpoint; do not overwrite foundation state.
aya_parent = torch.load(CHECKPOINT, map_location='cpu', weights_only=True)
task_target_step = int(aya_parent['step']) + TASK_SFT_STEPS
TASK_CHECKPOINT = CHECKPOINT_ROOT / 'artemus-task-sft.pt'
task_train_cmd = [
    'artemus-train',
    '--data', str(CAPABILITY_DIR / 'train.txt'),
    '--validation-data', str(CAPABILITY_DIR / 'validation.txt'),
    '--data-manifest', str(CAPABILITY_DIR / 'manifest.json'),
    '--document-format', 'text',
    '--tokenizer', 'sentencepiece',
    '--resume', str(CHECKPOINT),
    '--allow-data-change',
    '--phase', 'task_sft',
    '--response-only',
    '--steps', str(task_target_step),
    '--batch-size', str(BATCH_SIZE),
    '--gradient-accumulation-steps', str(GRAD_ACCUM),
    '--precision', PRECISION,
    '--override-resume-learning-rate',
    '--learning-rate', '0.000025',
    '--lr-schedule', 'cosine',
    '--min-learning-rate', '0.000003',
    '--warmup-steps', '50',
    '--eval-interval', '150',
    '--eval-iters', '8',
    '--seed', str(TRAIN_SEED + 17),
    '--device', 'auto',
    '--output', str(TASK_CHECKPOINT),
]
subprocess.run(task_train_cmd, check=True)
task_payload = torch.load(TASK_CHECKPOINT, map_location='cpu', weights_only=True)
assert int(task_payload['step']) == task_target_step
assert task_payload.get('response_only') is True
CHECKPOINT = TASK_CHECKPOINT
print('Task-SFT candidate:', CHECKPOINT, 'step:', task_payload['step'])

## 11. Basic generation and dialogue health check

These runs check model loading, non-empty generation, and short multi-turn context handling. They are health checks, not correctness benchmarks. A cycle is not a capability promotion; use protected exact-answer, code-test, and multilingual evaluations before serving a candidate.


In [ ]:
subprocess.run([
    'artemus-dialogue-eval', '--checkpoint', str(CHECKPOINT),
    '--scenarios', 'data/evaluation/dialogues.jsonl',
    '--output-dir', str(ARTIFACT_ROOT / 'dialogue-eval'),
    '--max-new-tokens', '64', '--top-k', '1', '--temperature', '1.0',
    '--seed', '1337', '--device', 'auto',
], check=True)
dialogue_summary = json.loads((ARTIFACT_ROOT / 'dialogue-eval' / 'summary.json').read_text())
print(json.dumps(dialogue_summary, indent=2))


## 12. Human teaching and supervised fine-tuning

You can talk to the model in a local terminal using `artemus-teach --checkpoint <checkpoint>`. It asks you to correct, approve, or skip each answer. Corrections are saved as human-reviewed candidates; model-generated answers are not included unless you explicitly enable the approval flag. Review the candidate JSONL before using it.

To use a reviewed candidate file as SFT data, upload it as a Kaggle input and run a separate stage with `--response-only --allow-data-change --allow-objective-change`. Do not combine arbitrary unreviewed chat logs with the training corpus.


## 13. Export checkpoint, all data-stage provenance, and run lineage

The export bundle contains the final candidate, any intermediate foundation/code checkpoints, the tokenizer, every available source and mixture manifest/provenance file, dialogue evaluation traces, the exact repository commit, the checkpoint's training lineage, and SHA-256 hashes for every exported artifact. Raw training corpora are not copied into the bundle. Keep the ZIP with the experiment record; do not commit model weights or datasets to Git.

In [ ]:
# Export an auditable, self-contained run bundle without raw corpora.
import hashlib
from datetime import datetime, timezone

RUN_STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
EXPORT_DIR = WORK_ROOT / f"artemus-run-{RUN_STAMP}"
EXPORT_DIR.mkdir(parents=True, exist_ok=False)

def sha256_path(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

# Preserve only stage checkpoints expected from this run; never sweep up arbitrary
# stale .pt files left in a reused notebook workspace.
checkpoint_paths = [
    CHECKPOINT_ROOT / "artemus-foundation-fineweb-edu.pt",
    CHECKPOINT_ROOT / "artemus-aya-instruction-sft.pt",
    CHECKPOINT_ROOT / "artemus-task-sft.pt",
]
if OSCR_MANIFEST_PATHS:
    checkpoint_paths.insert(1, CHECKPOINT_ROOT / "artemus-code-adapted.pt")
if CHECKPOINT.is_file() and CHECKPOINT not in checkpoint_paths:
    checkpoint_paths.append(CHECKPOINT)
for checkpoint_path in checkpoint_paths:
    if checkpoint_path.is_file():
        shutil.copy2(checkpoint_path, EXPORT_DIR / checkpoint_path.name)

# Keep each stage's manifest and provenance beside the weights they describe.
stage_sources = [
    ("fineweb-edu", DATA_DIR),
    ("fineweb2-bangla", FINEWEB2_DIR),
    ("foundation-mixture", MIX_DIR),
    ("aya-instruction-sft", AYA_DIR),
    ("oasst1", OASST_DIR),
    ("capability-sft", CAPABILITY_DIR),
]
if OSCR_MANIFEST_PATHS:
    stage_sources.append(("oscr-code", OSCR_DIR))

exported_source_files = []
for label, source_dir in stage_sources:
    if not source_dir.is_dir():
        raise RuntimeError(f"Expected training stage directory is missing: {source_dir}")
    for filename in ("manifest.json", "provenance.jsonl"):
        source_path = source_dir / filename
        if source_path.is_file():
            target_path = EXPORT_DIR / f"{label}-{filename}"
            shutil.copy2(source_path, target_path)
            exported_source_files.append(target_path.name)
    if not (EXPORT_DIR / f"{label}-manifest.json").is_file():
        raise RuntimeError(f"Missing exported source manifest for {label}")

# The final tokenizer and evaluation trace are part of the model's reproducibility record.
tokenizer_path = TOKENIZER_PREFIX.with_suffix(".model")
if not tokenizer_path.is_file():
    raise RuntimeError(f"Tokenizer model is missing: {tokenizer_path}")
shutil.copy2(tokenizer_path, EXPORT_DIR / "tokenizer.model")

dialogue_dir = ARTIFACT_ROOT / "dialogue-eval"
for filename in ("summary.json", "dialogues.jsonl"):
    source_path = dialogue_dir / filename
    if source_path.is_file():
        shutil.copy2(source_path, EXPORT_DIR / f"dialogue-{filename}")

# Preserve the resolved code revision and the checkpoint's complete training lineage.
checkpoint_payload = torch.load(CHECKPOINT, map_location="cpu", weights_only=True)
run_metadata = checkpoint_payload.get("run_metadata") or {}
run_summary = {
    "schema_version": 1,
    "repository": "https://github.com/FAHTHIZ/Artemus",
    "repository_commit": REPO_COMMIT,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "final_checkpoint": CHECKPOINT.name,
    "checkpoint_sha256": sha256_path(CHECKPOINT),
    "global_step": int(checkpoint_payload.get("step", 0)),
    "training_cycle": CYCLE_INDEX + 1,
    "cycle_seeds": {
        "fineweb_edu": FINEWEB_SHUFFLE_SEED,
        "fineweb2_bengali": FINEWEB2_SHUFFLE_SEED,
        "aya": AYA_SHUFFLE_SEED,
        "mixture": MIXTURE_SEED,
        "training": TRAIN_SEED,
    },
    "model_config": checkpoint_payload.get("model_config"),
    "run_metadata": run_metadata,
    "source_artifacts": sorted(exported_source_files),
}
(EXPORT_DIR / "run-metadata.json").write_text(
    json.dumps(run_summary, indent=2, ensure_ascii=False, default=str) + "\n",
    encoding="utf-8",
)

# Hash every exported file. The manifest intentionally does not hash itself.
artifact_records = []
for path in sorted(p for p in EXPORT_DIR.rglob("*") if p.is_file()):
    artifact_records.append({
        "path": path.relative_to(EXPORT_DIR).as_posix(),
        "size_bytes": path.stat().st_size,
        "sha256": sha256_path(path),
    })
export_manifest = {
    "schema_version": 1,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "repository_commit": REPO_COMMIT,
    "final_checkpoint": CHECKPOINT.name,
    "artifacts": artifact_records,
}
(EXPORT_DIR / "export-manifest.json").write_text(
    json.dumps(export_manifest, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)

EXPORT_ZIP = shutil.make_archive(str(WORK_ROOT / f"artemus-run-{RUN_STAMP}"), "zip", EXPORT_DIR)
print("Export directory:", EXPORT_DIR)
print("Export archive:", EXPORT_ZIP)
print("Exported files:", [p.name for p in sorted(EXPORT_DIR.iterdir())])
print("Data-stage provenance files:", exported_source_files)


## 14. Next steps

1. Compare the new foundation checkpoint against the old task-SFT checkpoint using the same evaluation set and decoding budget.
2. Add a curated Bangla stream only after reviewing its component/source rights. B-CORE's dataset-level CC-BY-4.0 label does not automatically clear every underlying source.
3. Add verified math/code SFT tasks using official training splits only; keep validation/test splits protected.
4. Add human-preference data and verified outcome optimization only after the basic supervised and tool-verification pipeline is stable.
5. Never train indefinitely on one sample: advance to a new pinned data release or stop when held-out validation plateaus or regresses.
